# 🎙️ The Synthetic Radio Host

**Transform Wikipedia Articles into Natural Hinglish Radio Conversations**

This notebook demonstrates how to:
1. Fetch content from Wikipedia
2. Generate natural Hinglish dialogue using AI
3. Convert the script to audio with distinct voices
4. Export as MP3

---


## 📦 Step 1: Install Dependencies


In [ ]:
# Install required packages
!pip install -q wikipedia-api google-generativeai edge-tts pydub nest-asyncio

# Install ffmpeg for audio processing
!apt-get install -qq ffmpeg

print("✅ All dependencies installed!")


## 🔑 Step 2: Set Up API Key

Get your free Gemini API key from: https://makersuite.google.com/app/apikey


In [ ]:
import os
from getpass import getpass

# Get API key securely
GEMINI_API_KEY = getpass("Enter your Gemini API Key: ")
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("✅ API Key configured!")


## 🏗️ Step 3: Import Libraries and Define Components


In [ ]:
# Enable async in notebooks
import nest_asyncio
nest_asyncio.apply()

import asyncio
import re
from pathlib import Path
from dataclasses import dataclass
from typing import List

import wikipediaapi
import google.generativeai as genai
import edge_tts
from pydub import AudioSegment as PydubSegment

print("✅ Libraries imported!")


In [ ]:
# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class ArticleData:
    """Wikipedia article data."""
    title: str
    summary: str
    key_facts: List[str]
    
    def get_content(self, max_words: int = 500) -> str:
        content = self.summary
        if self.key_facts:
            content += "\n\nKey Facts:\n" + "\n".join(f"- {f}" for f in self.key_facts[:5])
        words = content.split()
        if len(words) > max_words:
            content = " ".join(words[:max_words]) + "..."
        return content


@dataclass
class DialogueTurn:
    """Single dialogue turn."""
    speaker: str
    text: str
    
    def get_clean_text(self) -> str:
        """Remove action markers like [laughs]."""
        clean = re.sub(r'\[.*?\]', '', self.text)
        return re.sub(r'\s+', ' ', clean).strip()


@dataclass
class AudioSegmentData:
    """Generated audio segment."""
    file_path: Path
    speaker: str
    text: str

print("✅ Data classes defined!")


In [ ]:
# =============================================================================
# WIKIPEDIA FETCHER
# =============================================================================

class WikipediaFetcher:
    """Fetches Wikipedia article content."""
    
    def __init__(self):
        self.wiki = wikipediaapi.Wikipedia(
            user_agent='SyntheticRadioHost/1.0',
            language='en'
        )
    
    def fetch(self, topic: str) -> ArticleData:
        """Fetch Wikipedia article."""
        page = self.wiki.page(topic)
        
        if not page.exists():
            raise ValueError(f"Article not found: {topic}")
        
        key_facts = self._extract_facts(page.text)
        
        return ArticleData(
            title=page.title,
            summary=page.summary,
            key_facts=key_facts
        )
    
    def _extract_facts(self, text: str) -> List[str]:
        """Extract key facts from text."""
        facts = []
        sentences = re.split(r'(?<=[.!?])\s+', text)
        
        for sent in sentences[:50]:
            if len(sent) < 30 or len(sent) > 200:
                continue
            if re.search(r'\d{4}|\d+%|founded|won|first|largest', sent, re.I):
                facts.append(sent.strip())
                if len(facts) >= 8:
                    break
        
        return facts

print("✅ WikipediaFetcher ready!")


In [ ]:
# =============================================================================
# PROMPT ENGINEERING (THE SECRET SAUCE! 🌶️)
# =============================================================================

SYSTEM_PROMPT = """You are an expert scriptwriter for Indian FM radio shows. 
Your specialty is writing natural, engaging conversations in HINGLISH - 
a mix of Hindi and English written in Roman script.

## HINGLISH RULES:
1. Mix Hindi and English naturally in each sentence
2. Write ALL text in Roman script (no Devanagari)
3. Use common Hindi words: achcha, yaar, matlab, bilkul, dekho, sunno, hai na
4. English for technical terms, Hindi for emotions

## EXAMPLES OF GOOD HINGLISH:
- "Arey yaar, ye team ka history toh amazing hai!"
- "Matlab dekho, 2008 mein start hui thi, right?"
- "Achcha achcha, tell me more about this"

## CONVERSATION STYLE:
1. Use FILLERS: umm, err, achcha, basically, matlab, toh
2. Include INTERRUPTIONS marked as [interrupts]
3. Add REACTIONS: [laughs], [chuckles], haan haan, achcha achcha
4. Make it INFORMAL and friendly"""


def get_generation_prompt(article: ArticleData, word_count: int = 300) -> str:
    """Generate the prompt for script creation."""
    return f"""## YOUR TASK
Write a 2-minute radio conversation between two Indian hosts discussing: **{article.title}**

## HOST PROFILES
- **RAVI**: Senior host, knowledgeable, uses more Hindi. Warm and welcoming.
- **PRIYA**: Younger co-host, enthusiastic, asks good questions. Energetic.

## CONTENT TO DISCUSS
{article.get_content(max_words=500)}

## FORMAT REQUIREMENTS
1. Write exactly {word_count} words (±50 words)
2. Every line MUST start with: RAVI: or PRIYA:
3. Mark actions in brackets: [laughs], [interrupts], [chuckles]

## MANDATORY ELEMENTS:
1. At least 8 fillers: umm, achcha, matlab, basically, yaar
2. At least 3 [interrupts]
3. At least 4 reactions: [laughs], [chuckles]
4. At least 5 Hindi words per exchange

## NOW WRITE THE COMPLETE SCRIPT:"""

print("✅ Prompt engineering ready!")


In [ ]:
# =============================================================================
# COMPLETE PIPELINE - ALL-IN-ONE FUNCTION
# =============================================================================

# Voice configuration for distinct hosts
VOICES = {
    "RAVI": "en-IN-PrabhatNeural",   # Male Indian voice
    "PRIYA": "en-IN-NeerjaNeural"    # Female Indian voice
}


async def generate_radio_show_async(topic: str, duration_minutes: float = 2) -> tuple:
    """
    Generate a complete radio show from a Wikipedia topic.
    
    Args:
        topic: Wikipedia article topic (e.g., "Mumbai Indians")
        duration_minutes: Target duration in minutes
        
    Returns:
        Tuple of (output_path, script_text)
    """
    
    print(f"\n🎙️ Generating radio show for: {topic}")
    print("=" * 50)
    
    # Step 1: Fetch Wikipedia content
    print("📚 Step 1/5: Fetching Wikipedia content...")
    fetcher = WikipediaFetcher()
    article = fetcher.fetch(topic)
    print(f"   ✓ Found: {article.title}")
    
    # Step 2: Generate Hinglish script
    print("✍️ Step 2/5: Generating Hinglish script...")
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel(
        'models/gemini-1.5-pro',
        generation_config={'temperature': 0.8, 'top_p': 0.95, 'max_output_tokens': 2000}
    )
    
    word_count = int(duration_minutes * 150)
    prompt = f"{SYSTEM_PROMPT}\n\n{get_generation_prompt(article, word_count)}"
    response = model.generate_content(prompt)
    script = response.text.replace("```", "")
    print(f"   ✓ Generated {len(script.split())} words")
    
    # Step 3: Parse dialogue
    print("📝 Step 3/5: Parsing dialogue...")
    dialogues = []
    pattern = re.compile(r'^([A-Z]+)\s*:\s*(.+)$', re.MULTILINE)
    for line in script.split("\n"):
        line = line.strip()
        if not line:
            continue
        match = pattern.match(line)
        if match:
            dialogues.append(DialogueTurn(speaker=match.group(1), text=match.group(2)))
    print(f"   ✓ Found {len(dialogues)} dialogue turns")
    
    # Step 4: Convert to audio
    print("🔊 Step 4/5: Converting to audio...")
    output_dir = Path("audio_segments")
    output_dir.mkdir(exist_ok=True)
    
    segments = []
    for i, dialogue in enumerate(dialogues):
        text = dialogue.get_clean_text()
        if not text:
            continue
        
        voice = VOICES.get(dialogue.speaker, VOICES["RAVI"])
        output_path = output_dir / f"segment_{i:03d}.mp3"
        
        communicate = edge_tts.Communicate(text=text, voice=voice)
        await communicate.save(str(output_path))
        
        segments.append(AudioSegmentData(file_path=output_path, speaker=dialogue.speaker, text=text))
        print(f"   ✓ Segment {i+1}/{len(dialogues)}: {dialogue.speaker}")
    
    # Step 5: Stitch audio
    print("🎵 Step 5/5: Stitching audio...")
    pause = PydubSegment.silent(duration=300)  # 300ms pause
    combined = PydubSegment.empty()
    
    for segment in segments:
        if segment.file_path.exists():
            audio = PydubSegment.from_file(str(segment.file_path))
            if len(combined) > 0:
                combined += pause
            combined += audio
    
    # Normalize volume
    target_dBFS = -20.0
    change = target_dBFS - combined.dBFS
    combined = combined.apply_gain(change)
    
    # Export
    safe_name = topic.replace(' ', '_').lower()
    final_path = Path(f"{safe_name}_radio_show.mp3")
    combined.export(str(final_path), format="mp3", bitrate="192k")
    
    duration_seconds = len(combined) / 1000
    
    print("\n" + "=" * 50)
    print(f"✅ SUCCESS! Generated: {final_path}")
    print(f"   Duration: {duration_seconds:.1f} seconds ({duration_seconds/60:.1f} minutes)")
    print("=" * 50)
    
    return final_path, script, duration_seconds


def generate_radio_show(topic: str, duration_minutes: float = 2) -> tuple:
    """Synchronous wrapper for the async function."""
    return asyncio.run(generate_radio_show_async(topic, duration_minutes))

print("✅ Pipeline ready! Run the next cell to generate your radio show.")


## 🚀 Step 4: Generate Your Radio Show!

Run the cell below to generate a radio show about "Mumbai Indians":


In [ ]:
# 🎙️ GENERATE YOUR RADIO SHOW!
# Change the topic to any Wikipedia article you want

TOPIC = "Mumbai Indians"  # Try: "Shah Rukh Khan", "Taj Mahal", "Indian cuisine"

final_path, script, duration = generate_radio_show(TOPIC, duration_minutes=2)

# Display the script
print("\n📜 GENERATED SCRIPT:")
print("-" * 50)
print(script)


## 🎧 Step 5: Listen to Your Radio Show!


In [ ]:
from IPython.display import Audio, display

print("🎙️ YOUR SYNTHETIC RADIO SHOW IS READY!")
print("=" * 50)
print(f"Topic: {TOPIC}")
print(f"Duration: {duration:.1f} seconds ({duration/60:.1f} minutes)")
print("=" * 50)

# Play the audio
display(Audio(str(final_path)))


## 📥 Step 6: Download Your MP3


In [ ]:
from google.colab import files

# Download the generated MP3
files.download(str(final_path))
print("✅ Download started!")


## 🔄 Try Different Topics!

You can generate radio shows for any Wikipedia topic. Just change the `TOPIC` variable above and re-run the cells.

**Suggested Topics:**
- "Shah Rukh Khan"
- "Indian cuisine"  
- "Taj Mahal"
- "Cricket World Cup"
- "Bollywood"
- "Indian Premier League"

---

## 📝 100-Word Prompting Explanation

The Hinglish conversational style is achieved through **strategic prompt engineering**:

1. **Language Definition**: The prompt explicitly defines Hinglish as "Hindi and English in Roman script" with concrete examples like "Arey yaar" and "matlab dekho" to prevent pure Hindi/English outputs.

2. **Filler Enforcement**: Requiring "at least 8 fillers" (umm, achcha, basically) forces natural speech patterns that make dialogue sound authentic.

3. **Structural Markers**: [interrupts] and [laughs] tags create realistic conversation dynamics and enable TTS timing cues.

4. **Character Differentiation**: Distinct personalities (RAVI: senior/Hindi-heavy, PRIYA: young/English-heavy) ensure varied dialogue styles.

5. **Quantity Requirements**: Specific numbers (3 interruptions, 4 reactions) prevent shortcuts and ensure quality.

---

Built with ❤️ for AI Hackathons
